Cell 1. Purpose...


In [1]:
from pathlib import Path
ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / 'data').is_dir() and (p / 'experiments/clinical_protocols').is_dir()))
OUT = ROOT / 'experiments/external_benchmarks/initial_runs'
from pathlib import Path
import os
_REPOSITORY_ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / 'experiments/clinical_protocols').is_dir() and (p / 'data').is_dir()))
if os.name == 'nt' and (not str(_REPOSITORY_ROOT).startswith('\\\\?\\')):
    _REPOSITORY_ROOT = Path('\\\\?\\' + str(_REPOSITORY_ROOT))
_NOTEBOOK_DIR = _REPOSITORY_ROOT / 'experiments/external_benchmarks/initial_runs'
os.chdir(_NOTEBOOK_DIR)


In [2]:
import ast
from typing import Any, Dict, List, Optional
import numpy as np
import pandas as pd
MEDMCQA_PARQUET = ROOT / 'data/external_benchmark_sources/medmcqa/train-00000-of-00001.parquet'
PQA_LABELED_PARQUET = ROOT / 'data/external_benchmark_sources/pubmedqa/labeled/train-00000-of-00001.parquet'
PQA_UNLABELED_PARQUET = ROOT / 'data/external_benchmark_sources/pubmedqa/unlabeled/train-00000-of-00001.parquet'
MEDMCQA_OUT_CSV = OUT / 'medmcqa_prepared.csv'
PQA_LABELED_OUT_CSV = OUT / 'pubmedqa_labeled_prepared.csv'
PQA_UNLABELED_OUT_CSV = OUT / 'pubmedqa_unlabeled_prepared.csv'
medmcqa_raw = pd.read_parquet(MEDMCQA_PARQUET)
pqa_labeled_raw = pd.read_parquet(PQA_LABELED_PARQUET)
pqa_unlabeled_raw = pd.read_parquet(PQA_UNLABELED_PARQUET)
print('MedMCQA raw:', medmcqa_raw.shape)
print('PubMedQA labeled raw:', pqa_labeled_raw.shape)
print('PubMedQA unlabeled raw:', pqa_unlabeled_raw.shape)


MedMCQA raw: (182822, 11)
PubMedQA labeled raw: (1000, 5)
PubMedQA unlabeled raw: (61249, 4)


In [3]:
COP_TO_LETTER = {0: 'A', 1: 'B', 2: 'C', 3: 'D'}
LETTER_TO_OPCOL = {'A': 'opa', 'B': 'opb', 'C': 'opc', 'D': 'opd'}

def _clean(value: Any) -> str:
    if value is None:
        return ''
    if isinstance(value, float) and np.isnan(value):
        return ''
    return str(value).strip()
n_before = len(medmcqa_raw)
medmcqa_single = medmcqa_raw[medmcqa_raw['choice_type'] == 'single'].copy()
n_dropped_multi = n_before - len(medmcqa_single)
print(f'MedMCQA: dropped {n_dropped_multi} multi-answer rows out of {n_before} (kept {len(medmcqa_single)} single-answer rows).')
medmcqa_single['correct_option_letter'] = medmcqa_single['cop'].map(COP_TO_LETTER)
medmcqa_single = medmcqa_single[medmcqa_single['correct_option_letter'].notna()].copy()

def _correct_text(row: pd.Series) -> str:
    letter = row['correct_option_letter']
    return _clean(row[LETTER_TO_OPCOL[letter]])
medmcqa_single['correct_option_text'] = medmcqa_single.apply(_correct_text, axis=1)
medmcqa_out = pd.DataFrame({'case_id': medmcqa_single['id'].map(_clean), 'question': medmcqa_single['question'].map(_clean), 'option_a': medmcqa_single['opa'].map(_clean), 'option_b': medmcqa_single['opb'].map(_clean), 'option_c': medmcqa_single['opc'].map(_clean), 'option_d': medmcqa_single['opd'].map(_clean), 'correct_option_letter': medmcqa_single['correct_option_letter'], 'correct_option_text': medmcqa_single['correct_option_text'], 'explanation': medmcqa_single['exp'].map(_clean), 'subject_name': medmcqa_single['subject_name'].map(_clean), 'topic_name': medmcqa_single['topic_name'].map(_clean)})
medmcqa_out = medmcqa_out[(medmcqa_out['question'] != '') & (medmcqa_out['option_a'] != '') & (medmcqa_out['option_b'] != '') & (medmcqa_out['option_c'] != '') & (medmcqa_out['option_d'] != '')].reset_index(drop=True)
print('MedMCQA prepared:', medmcqa_out.shape)
medmcqa_out.head(3)


MedMCQA: dropped 62057 multi-answer rows out of 182822 (kept 120765 single-answer rows).


MedMCQA prepared: (120765, 11)


,case_id,question,option_a,option_b,option_c,option_d,correct_option_letter,correct_option_text,explanation,subject_name,topic_name
0,e9ad821a-c438-4965-9f77-760819dfa155,Chronic urethral obstruction due to benign pri...,Hyperplasia,Hyperophy,Atrophy,Dyplasia,C,Atrophy,Chronic urethral obstruction because of urinar...,Anatomy,Urinary tract
1,e3d3c4e1-4fb2-45e7-9f88-247cc8f373b3,Which vitamin is supplied from only animal sou...,Vitamin C,Vitamin B7,Vitamin B12,Vitamin D,C,Vitamin B12,Ans. (c) Vitamin B12 Ref: Harrison's 19th ed. ...,Biochemistry,Vitamins and Minerals
2,dc6794a3-b108-47c5-8b1b-3b4931577249,Growth hormone has its effect on growth through?,Directly,IG1-1,Thyroxine,Intranuclear receptors,B,IG1-1,"Ans. is 'b' i.e., IGI-1GH has two major functi...",Physiology,


In [4]:
def _as_list(value: Any) -> List[str]:
    if value is None:
        return []
    if isinstance(value, np.ndarray):
        return [str(x) for x in value.tolist()]
    if isinstance(value, (list, tuple)):
        return [str(x) for x in value]
    return [str(value)]

def build_context_text(context_value: Any) -> str:
    if context_value is None:
        return ''
    if isinstance(context_value, dict):
        contexts = _as_list(context_value.get('contexts'))
        labels = _as_list(context_value.get('labels'))
    else:
        return ''
    parts: List[str] = []
    for i, text in enumerate(contexts):
        text = text.strip()
        if not text:
            continue
        label = labels[i] if i < len(labels) else ''
        parts.append(f'{label}:\n{text}' if label else text)
    return '\n\n'.join(parts)

def build_meshes_text(context_value: Any) -> str:
    if not isinstance(context_value, dict):
        return ''
    return ' | '.join(_as_list(context_value.get('meshes')))

def build_context_labels_text(context_value: Any) -> str:
    if not isinstance(context_value, dict):
        return ''
    return ' | '.join(_as_list(context_value.get('labels')))

def prepare_pqa(df_raw: pd.DataFrame, include_final_decision: bool) -> pd.DataFrame:
    out = pd.DataFrame({'case_id': df_raw['pubid'].astype(str), 'question': df_raw['question'].map(_clean), 'context': df_raw['context'].map(build_context_text), 'context_labels': df_raw['context'].map(build_context_labels_text), 'meshes': df_raw['context'].map(build_meshes_text), 'long_answer': df_raw['long_answer'].map(_clean)})
    if include_final_decision:
        out['final_decision'] = df_raw['final_decision'].map(_clean)
    out = out[(out['question'] != '') & (out['context'] != '')].reset_index(drop=True)
    return out
pqa_labeled_out = prepare_pqa(pqa_labeled_raw, include_final_decision=True)
pqa_unlabeled_out = prepare_pqa(pqa_unlabeled_raw, include_final_decision=False)
print('PubMedQA labeled prepared:', pqa_labeled_out.shape)
print(pqa_labeled_out['final_decision'].value_counts(dropna=False))
print()
print('PubMedQA unlabeled prepared:', pqa_unlabeled_out.shape)
pqa_labeled_out.head(2)


PubMedQA labeled prepared: (1000, 7)
final_decision
yes      552
no       338
maybe    110
Name: count, dtype: int64

PubMedQA unlabeled prepared: (61249, 6)


,case_id,question,context,context_labels,meshes,long_answer,final_decision
0,21645374,Do mitochondria play a role in remodelling lac...,BACKGROUND:\nProgrammed cell death (PCD) is th...,BACKGROUND | RESULTS,Alismataceae | Apoptosis | Cell Differentiatio...,Results depicted mitochondrial dynamics in viv...,yes
1,16418930,Landolt C and snellen e acuity: differences in...,BACKGROUND:\nAssessment of visual acuity depen...,BACKGROUND | PATIENTS AND METHODS | RESULTS,"Adolescent | Adult | Aged | Aged, 80 and over ...","Using the charts described, there was only a s...",no


In [5]:
medmcqa_out.to_csv(MEDMCQA_OUT_CSV, sep=';', index=False, encoding='utf-8-sig')
pqa_labeled_out.to_csv(PQA_LABELED_OUT_CSV, sep=';', index=False, encoding='utf-8-sig')
pqa_unlabeled_out.to_csv(PQA_UNLABELED_OUT_CSV, sep=';', index=False, encoding='utf-8-sig')
print('Saved:')
print(f'  {MEDMCQA_OUT_CSV}: {medmcqa_out.shape}')
print(f'  {PQA_LABELED_OUT_CSV}: {pqa_labeled_out.shape}')
print(f'  {PQA_UNLABELED_OUT_CSV}: {pqa_unlabeled_out.shape}')


Saved:
  .\experiments\external_benchmarks\initial_runs\medmcqa_prepared.csv: (120765, 11)
  .\experiments\external_benchmarks\initial_runs\pubmedqa_labeled_prepared.csv: (1000, 7)
  .\experiments\external_benchmarks\initial_runs\pubmedqa_unlabeled_prepared.csv: (61249, 6)
